# SN impostor candidate search: whole DP2 footprint

**Author:** Mojgan Aghakhanloo (University of Virginia)
**Runs on:** Rubin Science Platform Notebook aspect, Data Preview 2

Searches every `DiaObject` in DP2 for eruptive massive stars (LBVs and SN impostors) in nearby galaxies.

**Stage 1 (inside the database, `DiaObject`):**
1. **Variability:** the total flux varies by at least `MIN_AMP_MAG` in r **or** g. This is written as a flux ratio, so no logarithm is needed in ADQL.

**Stage 2 (HECATE host galaxies, [Kovlakas et al. 2021](https://doi.org/10.1093/mnras/stab1799)):**

2. **Near a galaxy but off its nucleus:** projected offset between `OFFSET_MIN_KPC` and `OFFSET_MAX_KPC`, and at least `NUCLEUS_MIN_ARCSEC` from the centre (to avoid AGN).
3. **Luminosity:** peak absolute magnitude between -17 and -7.

Candidates with no catalogued host nearby are kept separately for inspection, because impostors in faint dwarfs would land there.

**Planned extensions:** require more than one peak; retrieve archival ZTF light curves of the final candidates; remove contaminants beyond the offset cut (red-noise AGN-like variability, inter-band lags, AGN catalogue matches, and spectroscopy, where a P Cygni profile confirms an impostor).

*Outputs are cleared because DP2 data are proprietary to Rubin data-rights holders.*

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

from lsst.rsp import RSPDiscovery
from lsst.utils.plotting import (get_multiband_plot_colors,
                                 get_multiband_plot_symbols)
from matplotlib.lines import Line2D


In [ ]:
discovery = RSPDiscovery("dp2")
service = discovery.get_tap_client()

filter_names   = ['u', 'g', 'r', 'i', 'z', 'y']
filter_colors  = get_multiband_plot_colors()
filter_symbols = get_multiband_plot_symbols()

def run_query(q):
    job = service.submit_job(q)
    job.run()
    job.wait(phases=['COMPLETED', 'ERROR'])
    if job.phase == 'ERROR':
        job.raise_if_error()
    return job.fetch_result().to_table()

## The only cell you edit

In [ ]:
VAR_BANDS = ['r', 'g']    # variability may occur in ANY of these bands
BAND = 'r'                 # band used for the peak absolute magnitude

# ---- quality ----
MIN_NDIASOURCES = 5        # min total detections across all bands
MIN_NDATA       = 5        # min detections in a variability band
MAX_SCIENCEFLUX = 1.45e6   # nJy; rejects very bright / saturated sources (AB mag < 16)

# ---- (1) magnitude variation ----
MIN_AMP_MAG = 1.0          # require total brightness to vary by >= this many mag

N_SHOW = 5                 # how many top candidates to plot

## Search the whole footprint

The cuts go into the `WHERE` clause, so only objects that pass come back.

**Total flux.** The `{b}_psfFlux*` columns are *difference-image* fluxes, measured relative to the template. The total flux at an epoch is therefore template + difference flux. The template is estimated as `scienceFluxMean - psfFluxMean`, the mean science-image flux minus the mean difference flux. Using `scienceFluxMean` alone would double-count the mean difference flux.

If the query is slow or returns too many objects, tighten the cuts (raise `MIN_AMP_MAG` or `MIN_NDATA`), or run it in RA strips (see `02_full_pipeline.ipynb`).

In [ ]:
amp_ratio   = 10 ** (MIN_AMP_MAG / 2.5)

col_set = {"ra", "dec", "diaObjectId", "nDiaSources", f"{BAND}_scienceFluxMean"}
for b in VAR_BANDS:
    col_set |= {f"{b}_scienceFluxMean", f"{b}_psfFluxMean", f"{b}_psfFluxNdata",
                f"{b}_psfFluxMax", f"{b}_psfFluxMin"}
cols = ", ".join(sorted(col_set))

def amp_condition(b):
    tmpl = f"({b}_scienceFluxMean - {b}_psfFluxMean)"          # template (quiescent) flux
    return (f"({b}_psfFluxNdata > {MIN_NDATA} AND ({tmpl} + {b}_psfFluxMin) > 0 AND "
            f"({tmpl} + {b}_psfFluxMax) > {amp_ratio:.6f} * ({tmpl} + {b}_psfFluxMin))")

conds = [
    f"nDiaSources > {MIN_NDIASOURCES}",
    f"{BAND}_scienceFluxMean > 0",
    f"{BAND}_scienceFluxMean < {MAX_SCIENCEFLUX}",
    "(" + " OR ".join(amp_condition(b) for b in VAR_BANDS) + ")",
]

query = f"SELECT {cols} FROM dp2.DiaObject WHERE " + " AND ".join(conds)
print(query)

## Run it and rank by amplitude

In [ ]:
DiaObjs = run_query(query)
# amplitude in magnitudes, per band, using template + difference flux
for b in VAR_BANDS:
    tmpl = DiaObjs[f'{b}_scienceFluxMean'] - DiaObjs[f'{b}_psfFluxMean']
    with np.errstate(invalid='ignore', divide='ignore'):
        DiaObjs[f'{b}_amp_mag'] = 2.5 * np.log10((tmpl + DiaObjs[f'{b}_psfFluxMax']) /
                                                 (tmpl + DiaObjs[f'{b}_psfFluxMin']))
DiaObjs['amp_mag'] = np.nanmax([np.asarray(DiaObjs[f'{b}_amp_mag'], float) for b in VAR_BANDS], axis=0)
DiaObjs['fluxamp'] = DiaObjs[f'{BAND}_psfFluxMax'] - DiaObjs[f'{BAND}_psfFluxMin']
DiaObjs.sort('amp_mag', reverse=True)
print(f"{len(DiaObjs)} candidates after Stage-1 cuts.")
# DiaObjs   # <- uncomment to view the table

## Stage 2: host galaxy and distance (criteria 2 and 3)

Cross-match each surviving object to **HECATE** (a nearby-galaxy catalogue with distances), then apply:
- **(2)** near a galaxy but off-nucleus: projected offset between `OFFSET_MIN_KPC` and `OFFSET_MAX_KPC`, and at least `NUCLEUS_MIN_ARCSEC` from the centre;
- **(3)** absolute magnitude between `M_BRIGHT` and `M_FAINT`.

**Why kpc rather than arcsec?** A fixed angle means different things at different distances: 10" is 0.5 kpc at 10 Mpc but 5 kpc at 100 Mpc. A projected offset in kpc treats all hosts the same. The arcsec guard is still needed, because positions closer than about the seeing (~1") cannot be separated from the nucleus whatever the distance.

**Choosing the values.** LBVs and impostors live in star-forming disks, typically a few kpc from the centre. The minimum offset should therefore only exclude the nuclear region (~0.5 kpc), not tens of kpc. The maximum (~50 kpc) keeps outer disks and satellites while rejecting chance alignments.

Each object is matched to the galaxy with the **smallest projected offset** among all HECATE galaxies within `OFFSET_MAX_KPC`, not simply the nearest one on the sky. A small background galaxy close in angle would otherwise steal the match from the large nearby galaxy that actually hosts the star.

Objects with **no** catalogued host within `OFFSET_MAX_KPC` are kept separately (`no_host`) for manual inspection.

In [ ]:
# ---- Stage 2 parameters ----
OFFSET_MIN_KPC     = 0.5   # (2) at least this far from the nucleus (rejects nuclear AGN)
OFFSET_MAX_KPC     = 50.0  # (2) but still plausibly associated with the galaxy
NUCLEUS_MIN_ARCSEC = 1.5   # (2) and resolved from the nucleus on the sky (~seeing)
M_BRIGHT = -17.0           # (3) absolute-mag window, brighter bound
M_FAINT  =  -7.0           # (3) fainter bound

In [ ]:
# HECATE = Heraklion Extragalactic Catalogue (Kovlakas+2021): ~200k galaxies < 200 Mpc with distances.
VIZIER_HECATE = "J/MNRAS/506/1896"
# Column names INSIDE HECATE -- VERIFY against the printout below, fix if different:
HECATE_RA, HECATE_DEC, HECATE_DIST = "RAJ2000", "DEJ2000", "D"

try:
    from astroquery.vizier import Vizier
    Vizier.ROW_LIMIT = -1
    cats = Vizier.get_catalogs(VIZIER_HECATE)
    hecate = next((t for t in cats if HECATE_DIST in t.colnames), cats[0])
except Exception as e:
    print("astroquery unavailable, using VizieR TAP instead:", e)
    import pyvo
    vtap = pyvo.dal.TAPService("https://tapvizier.cds.unistra.fr/TAPVizieR/tap")
    # if this errors on the table name, run:
    #   vtap.search(f\"SELECT table_name FROM TAP_SCHEMA.tables WHERE table_name LIKE '{VIZIER_HECATE}%'\").to_table()
    hecate = vtap.search(f'SELECT * FROM \"{VIZIER_HECATE}/table1\"').to_table()

print(len(hecate), "HECATE galaxies loaded")
print("columns:", hecate.colnames)
for c in (HECATE_RA, HECATE_DEC, HECATE_DIST):
    if c not in hecate.colnames:
        raise KeyError(f"'{c}' not in HECATE columns -- pick the right one from the list above")
hecate[:3]

In [ ]:
from astropy.coordinates import SkyCoord, search_around_sky

cand = SkyCoord(DiaObjs['ra'], DiaObjs['dec'], unit=u.deg)
D_all = np.asarray(hecate[HECATE_DIST], dtype=float)
okD = np.isfinite(D_all) & (D_all > 0)
gal = SkyCoord(hecate[HECATE_RA][okD], hecate[HECATE_DEC][okD], unit=u.deg)
D_gal = D_all[okD]

# all object-galaxy pairs within the largest angle OFFSET_MAX_KPC can subtend (capped at 3 deg)
max_deg = min(np.degrees(OFFSET_MAX_KPC / (D_gal.min() * 1e3)), 3.0)
i_c, i_g, sep, _ = search_around_sky(cand, gal, max_deg * u.deg)
off_kpc = D_gal[i_g] * 1e3 * sep.to(u.rad).value
keep = off_kpc <= OFFSET_MAX_KPC
i_c, i_g, sep, off_kpc = i_c[keep], i_g[keep], sep[keep], off_kpc[keep]

# best host = smallest projected offset for each object
order = np.lexsort((off_kpc, i_c))
i_c, i_g, sep, off_kpc = i_c[order], i_g[order], sep[order], off_kpc[order]
first = np.r_[True, i_c[1:] != i_c[:-1]]
i_c, i_g, sep, off_kpc = i_c[first], i_g[first], sep[first], off_kpc[first]

has_host = np.zeros(len(DiaObjs), bool); has_host[i_c] = True
no_host = DiaObjs[~has_host]                      # no catalogued host within OFFSET_MAX_KPC
matched = DiaObjs[i_c]
matched['host_D_Mpc']      = D_gal[i_g]
matched['host_sep_arcsec'] = sep.to(u.arcsec).value
matched['host_sep_kpc']    = off_kpc

# peak total flux = template + maximum difference flux -> absolute mag
b = BAND
tmpl = matched[f'{b}_scienceFluxMean'] - matched[f'{b}_psfFluxMean']
F_peak = np.asarray(tmpl + matched[f'{b}_psfFluxMax'], dtype=float)   # nJy
with np.errstate(invalid='ignore', divide='ignore'):
    m_peak = -2.5 * np.log10(F_peak) + 31.4
    matched['M_abs'] = m_peak - (5 * np.log10(matched['host_D_Mpc']) + 25)   # no extinction correction

near_but_outside = ((matched['host_sep_kpc'] > OFFSET_MIN_KPC) &
                    (matched['host_sep_arcsec'] > NUCLEUS_MIN_ARCSEC))
lum_ok = (matched['M_abs'] > M_BRIGHT) & (matched['M_abs'] < M_FAINT)

hosted = matched[near_but_outside & lum_ok]
hosted.sort('amp_mag', reverse=True)

print(f"{len(hosted)} impostor candidates: near a galaxy, off-nucleus, M in ({M_BRIGHT}, {M_FAINT})")
print(f"{len(no_host)} candidates with no catalogued host within {OFFSET_MAX_KPC} kpc (set aside for inspection)")
hosted['diaObjectId', 'host_D_Mpc', 'host_sep_kpc', 'host_sep_arcsec', 'M_abs', 'amp_mag']

> **Note on extinction:** `M_abs` here has no dust correction, so real objects can scatter fainter
> than they truly are (a genuine impostor could fall just below the −7 edge). For a proper sample,
> subtract at least the Galactic A_r (SFD/`dustmaps`) from `m_peak`; host extinction is unknown and is
> the dominant uncertainty: relevant since impostors are dusty.

## Pull light curves for the final candidates

These are the objects that passed **all** criteria (1–3), taken from `hosted`.

In [ ]:
if len(hosted) == 0:
    raise ValueError("No candidates passed all criteria (Stage 1 + Stage 2). Loosen the cuts.")

top = hosted[:N_SHOW]
cand_list = "(" + ", ".join(str(v) for v in top['diaObjectId']) + ")"
query = (
    "SELECT fsodo.diaObjectId, fsodo.visit, fsodo.band, "
    "fsodo.psfDiffFlux, fsodo.psfDiffFluxErr, "
    "fsodo.psfFlux, fsodo.psfFluxErr, "          # <- direct (science) image forced photometry
    "vis.expMidptMJD "
    "FROM dp2.ForcedSourceOnDiaObject AS fsodo "
    "JOIN dp2.Visit AS vis ON vis.visit = fsodo.visit "
    f"WHERE fsodo.diaObjectId IN {cand_list}"
)
FrcdSrc = run_query(query)
print(f"{len(FrcdSrc)} forced-source measurements retrieved.")

## Plot the final candidate light curves

Difference-image and direct-image flux (nJy) vs. MJD for the candidates that survived every cut. Convert a panel to AB mag with `(np.asarray(flux) * u.nJy).to(u.ABmag).value`.

In [ ]:
def plot_grid(ycol, ylabel, clip=False):
    cand_ids = list(top['diaObjectId'])
    n = len(cand_ids); ncol = 2; nrow = math.ceil(n / ncol)
    fig, ax = plt.subplots(nrow, ncol, figsize=(10, 2.4 * nrow), squeeze=False)
    for k in range(nrow * ncol):
        i, j = divmod(k, ncol); a = ax[i][j]
        if k >= n:
            a.set_axis_off(); continue
        oid = cand_ids[k]; lc = FrcdSrc[FrcdSrc['diaObjectId'] == oid]
        for filt in filter_names:
            m = lc['band'] == filt
            a.plot(lc['expMidptMJD'][m], lc[ycol][m],
                   filter_symbols[filt], ms=8, mew=0, alpha=0.6, color=filter_colors[filt])
        a.set_title(str(oid), fontsize=9)
        if clip:
            try:
                med, sd = np.nanmedian(lc[ycol]), np.nanstd(lc[ycol])
                a.set_ylim(med - 3 * sd, med + 3 * sd)
            except Exception:
                pass
        if i == nrow - 1: a.set_xlabel('MJD (days)')
        if j == 0: a.set_ylabel(ylabel)
    # one shared band legend for the whole figure
    handles = [Line2D([0], [0], marker=filter_symbols[f], color=filter_colors[f],
                      linestyle='none', markersize=8, label=f) for f in filter_names]
    fig.legend(handles=handles, loc='lower center', ncol=len(filter_names),
               frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.suptitle(ylabel, fontsize=11)
    plt.tight_layout(); plt.show()

plot_grid('psfDiffFlux', 'DiffIm flux (nJy)', clip=True)   # difference image
plot_grid('psfFlux',     'Direct flux (nJy)')              # science image